# SmartHire — Phase 3: Unsupervised Job Recommendation Engine
### Objective:
Build a content-based recommendation system that ranks job postings by computing **Cosine Similarity** between a candidate's CV and the pre-indexed vector space of job descriptions and required skill stacks.


In [1]:
import sys
from pathlib import Path
if str(Path("..").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("..").resolve()))

import pandas as pd
import numpy as np
import joblib
from src.models.recommender import JobRecommender
from src.features.text_features import load_vectorizer
from src.evaluate import evaluate_recommender_precision_at_k

# Ingest corpus
df = pd.read_csv("../data/interim/job_corpus_clean.csv")
import ast
df['parsed_skills'] = df['parsed_skills'].apply(lambda v: ast.literal_eval(v) if isinstance(v, str) and v.startswith('[') else [s.strip() for s in str(v).split(';') if s.strip()])

vectorizer = load_vectorizer("../models/tfidf_vectorizer.pkl")
recommender = JobRecommender(df, vectorizer=vectorizer)
print(f"Recommender Initialized with {len(df)} jobs and {len(recommender.skill_universe)} distinct skills.")


Recommender Initialized with 525 jobs and 113 distinct skills.


## 1. Sanity Check 1: Data Science Candidate CV
Testing recommendations on an analytical profile with Python, Machine Learning, and Tableau.


In [2]:
ds_cv = """
Senior Data Scientist with 5 years experience in machine learning, statistical modeling, and data visualization.
Technical Skills: Python, Scikit-learn, TensorFlow, Pandas, NumPy, SQL, Tableau, A/B Testing.
Experienced in training predictive models, feature engineering, and deploying data solutions.
"""

recs_ds = recommender.recommend(ds_cv, top_n=5)
rec_df_ds = pd.DataFrame(recs_ds)[['title', 'category', 'company', 'location', 'match_score', 'readiness_score', 'primary_gap']]
rec_df_ds


,title,category,company,location,match_score,readiness_score,primary_gap
0,Machine Learning Engineer,Data Science,"Snyder, Campos and Callahan",Pune,44.7,100.0,None
1,Data Scientist,Data Science,Burton Ltd,Delhi NCR,41.5,100.0,None
2,Data Analyst,Data Science,Blake and Sons,Noida,39.3,100.0,None
3,Data Analyst,Data Science,Doyle Ltd,Pune,39.0,100.0,None
4,Data Analyst,Data Science,Galloway-Wyatt,Pune,38.5,100.0,None


## 2. Sanity Check 2: DevOps & Cloud Engineer CV
Testing recommendations on an infrastructure and automation profile with AWS, Kubernetes, and Terraform.


In [3]:
devops_cv = """
Cloud Infrastructure Engineer with 4 years experience in CI/CD automation, cloud architecture, and container orchestration.
Core Skills: AWS, Docker, Kubernetes, Terraform, Ansible, Linux, Shell Scripting, Jenkins, Monitoring.
Built scalable Kubernetes clusters and automated infrastructure provisioning with Terraform.
"""

recs_devops = recommender.recommend(devops_cv, top_n=5)
rec_df_devops = pd.DataFrame(recs_devops)[['title', 'category', 'company', 'location', 'match_score', 'readiness_score', 'primary_gap']]
rec_df_devops


,title,category,company,location,match_score,readiness_score,primary_gap
0,Cloud Engineer,DevOps Engineer,Wells Inc,Chennai,47.4,83.3,Git
1,DevOps Engineer,DevOps Engineer,Reid-Poole,Delhi NCR,47.2,87.5,Git
2,Cloud Engineer,DevOps Engineer,Christensen PLC,Ahmedabad,45.7,87.5,Git
3,Cloud Engineer,DevOps Engineer,Curry Inc,Noida,45.5,87.5,Git
4,Cloud Engineer,DevOps Engineer,"Hernandez, Thompson and Boyd",Ahmedabad,42.4,85.7,Git


## 3. Sanity Check 3: UI/UX & Graphic Designer CV
Testing recommendations on a creative design profile with Figma, UI/UX, and Typography.


In [4]:
designer_cv = """
Creative Graphic Designer & UI/UX Specialist with expertise in digital product design and branding.
Proficient in Figma, Photoshop, Illustrator, InDesign, UI/UX, Typography, Branding, Creativity.
Crafted brand design systems, high-fidelity prototypes, and user research interfaces.
"""

recs_design = recommender.recommend(designer_cv, top_n=5)
rec_df_design = pd.DataFrame(recs_design)[['title', 'category', 'company', 'location', 'match_score', 'readiness_score', 'primary_gap']]
rec_df_design


,title,category,company,location,match_score,readiness_score,primary_gap
0,Graphic Designer,Graphic Designer,Moody-Taylor,Ahmedabad,50.2,100.0,None
1,Graphic Designer,Graphic Designer,King-Smith,Chennai,48.4,87.5,Communication
2,UI/UX Designer,Graphic Designer,"Andrews, Higgins and Carter",Noida,45.6,85.7,Communication
3,Graphic Designer,Graphic Designer,Wilkerson-Arias,Pune,43.5,100.0,None
4,UI/UX Designer,Graphic Designer,Burch-Montoya,Hyderabad,41.9,100.0,None


## 4. Recommender Precision@K Evaluation
Simulating Precision@5 across sample profiles:


In [5]:
p5_ds = evaluate_recommender_precision_at_k([r['category'] for r in recs_ds], 'Data Science', k=5)
p5_devops = evaluate_recommender_precision_at_k([r['category'] for r in recs_devops], 'DevOps Engineer', k=5)
p5_design = evaluate_recommender_precision_at_k([r['category'] for r in recs_design], 'Graphic Designer', k=5)

eval_df = pd.DataFrame([
    {"Profile": "Data Scientist", "Target Category": "Data Science", "Precision@5": f"{p5_ds*100:.0f}%"},
    {"Profile": "DevOps Engineer", "Target Category": "DevOps Engineer", "Precision@5": f"{p5_devops*100:.0f}%"},
    {"Profile": "Graphic Designer", "Target Category": "Graphic Designer", "Precision@5": f"{p5_design*100:.0f}%"}
])
eval_df


,Profile,Target Category,Precision@5
0,Data Scientist,Data Science,100%
1,DevOps Engineer,DevOps Engineer,100%
2,Graphic Designer,Graphic Designer,100%
